# Absorption spectrum of a gold nanowire

A gold wire of 50 nm radius in vacuum under a plane wave (electric field in the plane,
across the wire): the absorption and scattering cross-sections per unit length over the
visible spectrum. `hpfem.units` converts nanometres, `hpfem.materials` tabulates the
Johnson & Christy permittivity at every wavelength, and the plasmon resonance appears
near 500 nm.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import hpfem
from hpfem import units, materials

hpfem.set_log_level("warn")
radius = 50 * units.nm
gold = materials.get("Au")
print(gold)

In [ ]:
mesh = hpfem.square_with_disc(2, radius, half_width=4 * radius, outer=8 * radius, inclusion_tag=2)
dofs = hpfem.NedelecDofMap2D(mesh, 3)
surface = hpfem.Surface2D.around_cells(mesh, 2)
sides = [hpfem.box_tag.X_MIN, hpfem.box_tag.X_MAX, hpfem.box_tag.Y_MIN, hpfem.box_tag.Y_MAX]
print(mesh, dofs)

## Sweep

Every wavelength has its own permittivity and PML, so each one is a full solve
(the operator changes); the geometry and the space are shared.

In [ ]:
wavelengths = np.linspace(400, 800, 21) * units.nm
sigma_abs, sigma_sca = [], []
for lam in wavelengths:
    omega = units.angular_frequency(wavelength=lam)
    k0 = units.vacuum_wavenumber(omega)
    setup = hpfem.ScatteringSetup2D()
    setup.omega = omega
    setup.materials.set(2, gold.at(omega))
    setup.incident = hpfem.plane_wave([0.0, 1.0], [k0, 0.0])
    setup.formulation = hpfem.Formulation.SCATTERED_FIELD
    setup.pml = hpfem.PmlBox2D.uniform([-4 * radius, -4 * radius], [4 * radius, 4 * radius],
                                       4 * radius, k0, 1.0, hpfem.PmlProfile(2, 1e-10))
    setup.pec_tags = sides
    problem = hpfem.Scattering2D(dofs, setup)
    cs = hpfem.cross_sections(problem, problem.solve(), surface, 1.0)
    sigma_abs.append(cs.absorption)
    sigma_sca.append(cs.scattering)

In [ ]:
fig, ax = plt.subplots()
ax.plot(wavelengths / units.nm, np.array(sigma_abs) / (2 * radius), label="absorption")
ax.plot(wavelengths / units.nm, np.array(sigma_sca) / (2 * radius), label="scattering")
ax.set_xlabel("wavelength [nm]"); ax.set_ylabel("cross-section / diameter"); ax.legend(); ax.grid(alpha=0.3)
peak = wavelengths[int(np.argmax(sigma_abs))] / units.nm
print(f"absorption peak near {peak:.0f} nm")

## The same from a project file

`hpfem run` takes the equivalent JSON description; `wavelength.value` as a list runs the
sweep and the results come back as JSON (one entry per wavelength).

In [ ]:
from hpfem import project

spec = {
    "problem": "scattering", "dim": 2, "length_unit": "nm",
    "mesh": {"type": "square_with_disc", "n": 2, "radius": 50, "half_width": 200, "outer": 400},
    "order": 3,
    "wavelength": {"value": [450, 500, 550], "unit": "nm"},
    "materials": {"background": "vacuum", "2": "Au"},
    "source": {"type": "plane_wave", "angle": 0, "polarisation": "TE"},
    "pml": {"lower": [-200, -200], "upper": [200, 200], "thickness": 200,
            "profile": {"order": 2, "reflection": 1e-10}},
    "boundaries": {"pec": ["x_min", "x_max", "y_min", "y_max"]},
    "outputs": {"cross_sections": {"around_tag": 2}},
}
for entry in project.run(spec)["results"]:
    print(f"{entry['wavelength_nm']:.0f} nm: sigma_abs = {entry['cross_sections']['absorption']:.3e} m")